# 14. Final submission workflow
This notebook documents the complete audio-to-score pipeline: Whisper base transcription, original linguistic features, acoustic/prosodic features, feature fusion, validated model selection, training RMSE, and test prediction generation.

The expected final output is `outputs/test_predictions.csv`, preserving all 216 rows and filenames from `test.csv`.


## Architecture
Audio -> Whisper ASR (base, fp16=False, CUDA when available) -> transcript features + acoustic features -> 50/50 Ridge(alpha=30) and HistGradientBoostingRegressor -> predicted grammar score.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
train = pd.read_csv(ROOT / 'train.csv')
test = pd.read_csv(ROOT / 'test.csv')
results = pd.read_csv(ROOT / 'outputs' / 'experiment_results.csv')
print('train rows:', len(train), 'test rows:', len(test))
print(results.sort_values('rmse').head(10).to_string(index=False))

In [ ]:
predictions_path = ROOT / 'outputs' / 'test_predictions.csv'
if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    print(predictions.shape)
    predictions['prediction'].hist(bins=20)
    plt.xlabel('Predicted grammar score')
    plt.show()
else:
    print('Run scripts/predict_test.py to transcribe test audio and create predictions.')

## Test inference and submission contract

The test workflow uses the current `test.csv` as the sole source of test filenames. It reuses `scripts/predict_test.py`, which transcribes test audio with Whisper, extracts the original linguistic and acoustic features, loads `outputs/final_model.joblib`, and writes continuous predictions to `outputs/test_predictions.csv`.

The submission contract is checked before writing `outputs/submission.csv`: exactly 216 predictions, no missing values, filenames in exactly the same order as `test.csv`, and every prediction within the valid 1–5 score range. `sample_submission.csv` is intentionally not used to define test IDs because its filename set and row count differ.

In [ ]:
# Run the existing reusable inference pipeline when this notebook is executed.
# This call is intentionally not run during documentation validation because it may invoke Whisper.
from scripts.predict_test import main as generate_test_predictions

generate_test_predictions()

predictions = pd.read_csv(ROOT / "outputs" / "test_predictions.csv")
assert len(predictions) == len(test) == 216
assert predictions["prediction"].notna().all()
assert predictions["filename"].tolist() == test["filename"].tolist()
assert predictions["prediction"].between(1, 5).all()

submission = predictions.rename(columns={"prediction": "label"})[["filename", "label"]]
submission.to_csv(ROOT / "outputs" / "submission.csv", index=False)

print(submission.head())
print(submission["label"].describe())
print("Validated predictions:", len(submission))
print("Saved:", ROOT / "outputs" / "submission.csv")

## Final conclusion

When the inference cell completes, `outputs/submission.csv` contains the current test filenames and continuous predicted labels, with no rounding applied. The assertions make the file ready for submission only when row count, alignment, missingness, and score-range requirements all pass.